# AMPR Phase 3 — Precompute Contact Maps (Test Split) on Colab T4

Run on **Google Colab** (~1-2h). Downloads PDB structures for test-split proteins,
computes Cα–Cα contact maps, and merges with the train+valid cmap from Phase 2
into `cmap_all.h5`.

Prerequisites:
- Phase 2 `cmap_train_valid.h5` available on Drive or Kaggle dataset `ampr-pdbch`
- `splits.json` and `protein_order.json` in `data/pdbch/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/YOUR_USERNAME/datn /content/datn
%cd /content/datn
!pip install -q biopython==1.84 h5py tqdm pyyaml

In [ ]:
import os
os.makedirs('data/contact_maps', exist_ok=True)
os.makedirs('data/pdb_cache', exist_ok=True)

# Option A: copy from Drive
import shutil
shutil.copy('/content/drive/MyDrive/ampr/cmap_train_valid.h5', 'data/contact_maps/cmap_train_valid.h5')

# Option B: from Kaggle (uncomment if using Kaggle dataset)
# !ln -sf /kaggle/input/ampr-pdbch/cmap_train_valid.h5 data/contact_maps/cmap_train_valid.h5

In [ ]:
# Precompute cmap for test split and merge into cmap_all.h5
!python scripts/precompute_cmap_test.py \
  --split data/pdbch/splits.json \
  --split_key test \
  --protein_order data/pdbch/protein_order.json \
  --train_valid_h5 data/contact_maps/cmap_train_valid.h5 \
  --out data/contact_maps/cmap_all.h5 \
  --pdb_dir data/pdb_cache/

In [ ]:
# Verify merge
import h5py
with h5py.File('data/contact_maps/cmap_all.h5', 'r') as f:
    print(f'cmap_all.h5 keys: {len(f.keys())} proteins')

In [ ]:
# Upload merged cmap to Kaggle
!pip install -q kaggle
import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

os.makedirs('/content/upload_cmap', exist_ok=True)
shutil.copy('data/contact_maps/cmap_all.h5', '/content/upload_cmap/')

from datetime import datetime
msg = f'phase3 cmap_all {datetime.utcnow().isoformat()}'
!kaggle datasets version -p /content/upload_cmap -m "{msg}" --dir-mode zip
print('Upload complete.')